# Paperless-ngx ML Platform — Chameleon Provisioning

Solo build. Brings up the full integrated ML system on a single VM:

- Paperless-ngx (custom fork with HTR + semantic-search UI)
- Data stack: PostgreSQL × 2, MinIO, Redpanda, Qdrant, Adminer
- ML serving: **ml_gateway** (pretrained TrOCR + mpnet)
- **qdrant_indexer** (chunks merged_text, populates Qdrant)
- **drift_monitor** (online MMD against IAM reference)
- **htr_consumer** (Kafka→slice→HTR→DB)
- Observability: Prometheus, Grafana (pre-provisioned dashboards), Alertmanager, MLflow, rollback controller

**Pre-reqs on your local machine before running this notebook:**
- `ghcr.io/redes01/paperless-ngx-ml:latest` has been pushed to GHCR (via `scripts/build_and_push.ps1`). If you've already pushed once and haven't changed the fork, skip.
- `paperless_data` and `paperless_data_integration` repos on GitHub are up to date.

Running this notebook provisions the VM, installs Docker, clones both repos, builds every service image on the VM, and starts everything. Expect **30–45 minutes total bring-up** on first run (most of which is pip installing torch + transformers for ml_gateway and qdrant_indexer).

---
## Part 1 — VM Setup

### Step 1 — Select project and site

In [ ]:
from chi import server, context, lease, network
import chi, os, datetime, time
from datetime import timedelta, timezone

context.version = "1.0"
context.choose_project()
# CHI@TACC gives bare-metal nodes with 48+ cores and hundreds of GB of
# local disk — much more headroom than KVM@TACC's m1.xlarge (8 vCPU, 40 GB).
# Training TrOCR fine-tunes and hosting 10+ containers both want the disk.
context.choose_site(default="CHI@TACC")
username = os.getenv("USER")
print(f"Username: {username}")


### Step 2 — Reserve a bare-metal node for 7 days

Node type: `compute_cascadelake` on CHI@TACC — 2× Intel Xeon Gold 6240R (48 cores total), 192 GB RAM, ~960 GB local SSD. Plenty of room for the full stack (Paperless, MinIO, Redpanda, Qdrant, MLflow, Grafana, Prometheus, ml_gateway with TrOCR + mpnet loaded, drift monitor, training runs) without disk-pressure games.

Lease runs for **7 days** from when you execute the cell below. If the demo slips and you need more time, extend the active lease from the [Chameleon web UI](https://chi.tacc.chameleoncloud.org/) — no need to tear down and re-provision.


In [ ]:
# 7-day lease from now. Extend via the Chameleon portal if the demo slips:
#   https://chi.tacc.chameleoncloud.org/
lease_duration = timedelta(days=7)
lease_end = datetime.datetime.now(timezone.utc) + lease_duration
print(f"Lease duration: {lease_duration}  (end: {lease_end.isoformat()})")

l = lease.Lease(
    f"lease-paperless-integration-{username}",
    duration=lease_duration,
)
# CHI bare metal: reserve by node_type, not flavor.
l.add_node_reservation(node_type="compute_cascadelake", amount=1)
# Also reserve a floating IP on CHI (on KVM, FIPs auto-allocate; on CHI
# they come from the project pool and must be requested via the lease).
l.add_fip_reservation(amount=1)
l.submit(idempotent=True)
print("Waiting for lease to become ACTIVE (can take a few minutes)...")
l.wait()
l.show()


### Step 3 — Launch the reserved node

On CHI bare metal, the `Server` uses `reservation_id` (not `flavor_name`). Provisioning takes ~5-10 min — the node physically powers on and PXE-boots the image.


In [ ]:
# Find an SSH key to attach to the node. CHI@TACC stores keys per-site,
# separate from KVM@TACC. If the project doesn't have one yet, upload your
# public key at:  https://chi.tacc.chameleoncloud.org/project/key_pairs/
# then re-run this cell.
import chi
_nova = chi.nova()
_existing_keys = [_kp.name for _kp in _nova.keypairs.list()]
if not _existing_keys:
    raise RuntimeError(
        "No SSH keypairs uploaded to this CHI@TACC project. "
        "Upload one at https://chi.tacc.chameleoncloud.org/project/key_pairs/ "
        "(Import Public Key from your ~/.ssh/id_rsa.pub or id_ed25519.pub) "
        "and re-run this cell."
    )
# Prefer a key whose name contains the current username; else take the first.
_key_name = next(
    (k for k in _existing_keys if username.lower() in k.lower()),
    _existing_keys[0],
)
print(f"Using SSH key: {_key_name}  (all keys in project: {_existing_keys})")

# Grab the node reservation id from the active lease.
# python-chi returns reservations as dicts, not objects: use ["id"] not .id.
node_reservation_id = l.node_reservations[0]["id"]

s = server.Server(
    f"node-paperless-integration-{username}",
    reservation_id=node_reservation_id,
    image_name="CC-Ubuntu24.04",
    key_name=_key_name,
)
s.submit(idempotent=True)
print("Node provisioning — takes 5-10 min on bare metal (PXE boot + image write)...")
s.wait()
print("Node ready.")


### Step 4 — Assign floating IP

The FIP was reserved as part of the lease in Step 2. `associate_floating_ip()` picks one from the lease's pool and attaches it.


In [ ]:
s.associate_floating_ip()
s.refresh()
s.show(type="widget")

### Step 5 — Open security groups

Consolidated into **two** groups so we stay well under Chameleon's 10-per-project quota (even after prior runs leave orphan groups behind). Re-running this step is safe: existing groups get reused, nothing duplicates.

If you hit `OverQuotaClient: Quota exceeded for resources: ['security_group']`, the block below first detaches + deletes orphan `allow-*` groups from prior sessions, then creates the two we actually need.

Ports bundled in `allow-paperless`:
- `8000` Paperless UI, `8080` Airflow, `5050` Adminer
- `9000` MinIO API, `9001` MinIO console
- `8090` Redpanda console, `6333` Qdrant
- `3000` Grafana, `9090` Prometheus, `9093` Alertmanager
- `5000` MLflow


In [ ]:
# ── Step 5a: free quota by deleting orphan security groups from prior runs ──
#
# Chameleon caps the project at 10 security groups per site. Old groups
# from previous runs count against the quota even when no node uses
# them. Clear anything non-default before we create the two we need.
#
# We use the nova compute API for attached-SG listing + detach, because
# python-chi's Server object doesn't expose get_security_groups(). Nova
# returns security_groups as a list of {"name": ...} dicts on the server
# resource object.
_neutron = chi.neutron()
_nova    = chi.nova()

# Step 1 — detach all non-default groups from THIS server so they\'re
# deletable. If the server doesn\'t report any attached groups (e.g.,
# API version differences), we skip straight to the delete loop below.
try:
    _nova_server = _nova.servers.get(s.id)
    _attached = list(getattr(_nova_server, "security_groups", []) or [])
except Exception as _exc:
    print(f"could not list attached security groups via nova: {_exc}")
    _attached = []

for _sg in _attached:
    _name = _sg["name"] if isinstance(_sg, dict) else getattr(_sg, "name", None)
    if not _name or _name == "default":
        continue
    try:
        _nova_server.remove_security_group(_name)
    except Exception:
        pass  # best-effort; delete loop below will catch what can be freed

# Step 2 — delete every non-default group in the project. Groups still
# in use by other servers/resources raise 409 Conflict, which we swallow.
for _g in _neutron.list_security_groups()["security_groups"]:
    if _g["name"] == "default":
        continue
    try:
        _neutron.delete_security_group(_g["id"])
    except Exception:
        pass  # in-use elsewhere; leave it

_remaining = len(_neutron.list_security_groups()["security_groups"])
print(f"security groups in project: {_remaining} / 10")

# ── Step 5b: create the two consolidated groups and attach them ──
security_groups = [
    {"name": "allow-ssh",       "ports": [22],
     "description": "SSH"},
    {"name": "allow-paperless", "ports": [8000, 8080, 5050, 9000, 9001,
                                           8090, 6333, 3000, 9090, 9093, 5000],
     "description": "Paperless + ML stack ports"},
]

# Re-read what\'s attached after the detach pass above — gives us the
# ground-truth for the idempotent attach loop that follows.
try:
    _nova_server = _nova.servers.get(s.id)
    attached = {
        (_sg["name"] if isinstance(_sg, dict) else getattr(_sg, "name", None))
        for _sg in (getattr(_nova_server, "security_groups", []) or [])
    }
except Exception:
    attached = set()

for sg in security_groups:
    secgroup = network.SecurityGroup({"name": sg["name"], "description": sg["description"]})
    for p in sg["ports"]:
        secgroup.add_rule(direction="ingress", protocol="tcp", port=p)
    secgroup.submit(idempotent=True)
    if sg["name"] not in attached:
        try:
            s.add_security_group(sg["name"])
            attached.add(sg["name"])
        except Exception as exc:
            if "Duplicate" not in str(exc):
                raise

print(f"Ensured {len(security_groups)} security groups on the server")


### Step 6 — Install Docker

In [ ]:
s.refresh()
s.check_connectivity()
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
print("Docker installed.")

---
## Part 2 — Deploy the stack

### Step 7 — Clone repos

Both from `REDES01`. `paperless_data` contains data pipelines + quality scripts + drift reference builder; `paperless_data_integration` contains every service compose.

In [ ]:
DATA_REPO        = "https://github.com/REDES01/paperless_data.git"
INTEGRATION_REPO = "https://github.com/REDES01/paperless_data_integration.git"

s.execute("rm -rf ~/paperless_data ~/paperless_data_integration")
s.execute(f"git clone {DATA_REPO} ~/paperless_data")
s.execute(f"git clone {INTEGRATION_REPO} ~/paperless_data_integration")
s.execute("ls ~/")
print("Repos cloned.")

### Step 8 — Write Paperless secret key

Must run this before bringing up the Paperless stack.

In [ ]:
s.execute(
    "cd ~/paperless_data_integration/paperless && "
    "cp docker-compose.env.example docker-compose.env && "
    "SECRET=$(python3 -c 'import secrets; print(secrets.token_urlsafe(64))') && "
    "sed -i \"s|PAPERLESS_SECRET_KEY=replace-me-with-a-real-secret|PAPERLESS_SECRET_KEY=$SECRET|\" docker-compose.env && "
    "grep PAPERLESS_SECRET_KEY docker-compose.env"
)
print("Secret key written.")

### Step 9 — Pull Paperless image from GHCR

In [ ]:
s.execute("sg docker -c 'docker pull ghcr.io/redes01/paperless-ngx-ml:latest'")
print("Paperless image pulled.")

### Step 10 — Bring up data stack + apply ML schema migration

Starts PostgreSQL (data-stack), MinIO, Redpanda, Qdrant, Adminer. Then applies the `paperless_doc_id` migration idempotently so the HTR consumer's upserts work.

In [ ]:
s.execute("cd ~/paperless_data_integration && sg docker -c 'bash scripts/create_network.sh'")
s.execute("cd ~/paperless_data_integration && sg docker -c 'bash scripts/up_paperless_data.sh'")

print("Waiting 25s for postgres + minio to become healthy...")
time.sleep(25)

# paperless_doc_id migration (idempotent)
s.execute(
    "cat ~/paperless_data_integration/seed/phase2_add_paperless_doc_id.sql | "
    "sg docker -c 'docker exec -i postgres psql -U user -d paperless'"
)
print("Data stack up, ML schema migrated.")

### Step 11 — Bring up Paperless

In [ ]:
s.execute("cd ~/paperless_data_integration && sg docker -c 'bash scripts/up_paperless.sh'")
print("Waiting 45s for Paperless to finish starting...")
time.sleep(45)
s.execute("sg docker -c 'docker ps --filter name=paperless --format \"{{.Names}}\t{{.Status}}\"'")

### Step 12 — Create Paperless superuser + generate API token

In [ ]:
# Create superuser (admin/admin)
s.execute(
    "sg docker -c 'docker exec paperless-webserver-1 python manage.py shell -c \""
    "from django.contrib.auth.models import User; "
    "User.objects.filter(username=\\\"admin\\\").exists() or "
    "User.objects.create_superuser(\\\"admin\\\", \\\"admin@example.com\\\", \\\"admin\\\"); "
    "print(\\\"Superuser ready\\\")\"'"
)

# Fetch API token for admin (NOT User.objects.first(), which returns AnonymousUser)
result = s.execute(
    "sg docker -c 'docker exec paperless-webserver-1 python manage.py shell -c \""
    "from rest_framework.authtoken.models import Token; "
    "from django.contrib.auth.models import User; "
    "t, _ = Token.objects.get_or_create(user=User.objects.get(username=\\\"admin\\\")); "
    "print(t.key)\"'"
)
PAPERLESS_TOKEN = result.stdout.strip().split("\n")[-1]
print(f"API Token: {PAPERLESS_TOKEN}")

### Step 13 — Build and start all ML services

This step builds images on the VM for:
- `ml_gateway` (TrOCR + mpnet) — **longest, ~15 min** on first build (pip install torch + transformers + pre-download models)
- `qdrant_indexer` (mpnet cached from earlier build)
- `drift_monitor`
- `htr_consumer`
- `observability` stack (prometheus, alertmanager, grafana, mlflow, rollback_ctrl)

All containers join the shared `paperless_ml_net`. Grafana dashboards (drift + ml_gateway) are pre-provisioned from `observability/grafana/dashboards/*.json` — no manual import needed.

In [ ]:
print("Building and starting ml_gateway, qdrant_indexer, drift_monitor, observability, htr_consumer...")
print("First build is ~15 min (downloads + caches TrOCR + mpnet into ml_gateway image)")
print()

# up_all.sh orchestrates all the compose ups in dependency order
s.execute(
    f"cd ~/paperless_data_integration && "
    f"PAPERLESS_TOKEN={PAPERLESS_TOKEN} "
    f"sg docker -c 'bash scripts/up_all.sh'"
)

print("\nVerifying observability stack…")
# Grafana dashboards are provisioned via bind-mount from
# ~/paperless_data_integration/observability/grafana/dashboards/ — both dashboards
# should show up in the /api/search response within ~30s of grafana boot.
# Also confirms Prometheus has metrics for drift_monitor + ml_gateway — if the
# datasource name in a dashboard JSON ever drifts out of sync (e.g. case mismatch
# with the provisioned datasource), panels silently show "No data" without errors,
# so this check catches that class of regression early.
s.execute(
    "sg docker -c 'docker exec paperless-webserver-1 curl -sf -u admin:admin "
    "http://grafana:3000/api/search?type=dash-db | python3 -m json.tool' 2>&1 | "
    "head -40"
)

s.execute(
    "sg docker -c 'docker exec paperless-webserver-1 curl -sf "
    "http://prometheus:9090/api/v1/query?query=drift_events_total | python3 -m json.tool' "
    "2>&1 | head -15"
)


### Step 14 — Wait for ml_gateway to become healthy

First boot loads TrOCR + mpnet into RAM (~60s). Also give Paperless + other services time to finish warming up.

In [ ]:
print("Waiting 90s for ml_gateway startup + model load...")
time.sleep(90)

s.execute("sg docker -c 'docker ps --format \"table {{.Names}}\t{{.Status}}\"'")
print()
print("ml_gateway health check:")
s.execute("curl -sf http://localhost:8100/health || echo 'not ready yet'")

### Step 15 — Upload sample documents

Uploads the two committed samples. Paperless ingestion is async (Tesseract OCR + thumbnail); wait ~60s before looking up IDs.

In [ ]:
s.execute(
    f"for f in ~/paperless_data_integration/sample_documents/sample_budget_memo.pdf "
    f"~/paperless_data_integration/sample_documents/sample_scan.jpeg; do "
    f"echo \"Uploading $f...\"; "
    f"curl -s -X POST http://localhost:8000/api/documents/post_document/ "
    f"-H \"Authorization: Token {PAPERLESS_TOKEN}\" "
    f"-F \"document=@$f\"; "
    f"echo; done"
)
print("Uploads submitted. Waiting 60s for Paperless to ingest...")
time.sleep(60)

---
## Part 3 — Drift reference + data quality artifacts

This part produces the artifacts that the drift dashboard and training-set manifest reference in the demo video.


### Step 19 — Ingest IAM (needed by drift reference + training baseline)

This populates `s3://paperless-datalake/warehouse/iam_dataset/` with Parquet shards. Takes a couple minutes.

In [ ]:
s.execute(
    "cd ~/paperless_data && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "-e MINIO_ENDPOINT=minio:9000 "
    "-e MINIO_ACCESS_KEY=admin "
    "-e MINIO_SECRET_KEY=paperless_minio "
    "python:3.12-slim bash -c \""
    "pip install -q pyarrow minio datasets tqdm Pillow && "
    "python ingestion/ingest_iam.py\"'"
)
print("IAM ingestion complete.")

### Step 20 — Run ingestion validation (Point 1)

Produces `s3://paperless-datalake/warehouse/iam_dataset/_validation/<ts>.json`.

In [ ]:
s.execute(
    "cd ~/paperless_data && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "-e MINIO_ENDPOINT=minio:9000 "
    "-e MINIO_ACCESS_KEY=admin "
    "-e MINIO_SECRET_KEY=paperless_minio "
    "python:3.12-slim bash -c \""
    "pip install -q -r batch_pipeline/requirements.txt && "
    "python batch_pipeline/validate_ingestion.py\"'"
)

### Step 21 — Build the drift reference

Fits `MMDDriftOnline` on 500 IAM crops and saves to `s3://paperless-datalake/warehouse/drift_reference/htr_v1/cd/`. drift_monitor's background retry loop picks up the detector automatically within 60 seconds — no restart needed.

In [ ]:
s.execute(
    "cd ~/paperless_data && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "-e MINIO_ENDPOINT=minio:9000 "
    "-e MINIO_ACCESS_KEY=admin "
    "-e MINIO_SECRET_KEY=paperless_minio "
    "python:3.12-slim bash -c \""
    "pip install --no-cache-dir --index-url https://download.pytorch.org/whl/cpu torch==2.4.1 && pip install --no-cache-dir alibi-detect pyarrow minio Pillow && "
    "python scripts/build_drift_reference.py\"'"
)


print("Reference detector uploaded to MinIO.")
print("drift_monitor's background retry loop will load it within 60 seconds.")


### Step 22 — Upload OOD samples for the drift demo

In [ ]:
s.execute(
    "cd ~/paperless_data/scripts && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "python:3.12-slim bash -c \""
    "pip install -q faker Pillow minio && "
    "python make_ood_samples.py && "
    "MINIO_ENDPOINT=minio:9000 MINIO_ACCESS_KEY=admin MINIO_SECRET_KEY=paperless_minio "
    "python upload_ood_to_minio.py\"'"
)

---
## Part 4 — Access URLs + API token

In [ ]:
s.refresh()
floating_ip = None
for net, addrs in (s.addresses or {}).items():
    for addr in addrs:
        if addr.get("OS-EXT-IPS:type") == "floating":
            floating_ip = addr["addr"]

print(f"Floating IP: {floating_ip}")
print()
print("═" * 70)
print("  Service URLs")
print("═" * 70)
print(f"  Paperless UI        http://{floating_ip}:8000           admin / admin")
print(f"  HTR review          http://{floating_ip}:8000/ml/htr-review")
print(f"  Semantic search     http://{floating_ip}:8000/ml/search")
print(f"  Grafana             http://{floating_ip}:3000           admin / admin")
print(f"  Prometheus          http://{floating_ip}:9090")
print(f"  Alertmanager        http://{floating_ip}:9093")
print(f"  MLflow              http://{floating_ip}:5000")
print(f"  MinIO Console       http://{floating_ip}:9001           admin / paperless_minio")
print(f"  Adminer             http://{floating_ip}:5050           user / paperless_postgres")
print(f"  Redpanda Console    http://{floating_ip}:8090")
print(f"  Qdrant Dashboard    http://{floating_ip}:6333/dashboard")
print(f"  Airflow             http://{floating_ip}:8080           admin / admin")
print()
print(f"  API Token: {PAPERLESS_TOKEN}")
print(f"  SSH:       ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")

---
## Part 5 — Training pipeline bootstrap

Seeds the MLflow `htr_training` experiment with a baseline + one fine-tuned candidate so the model registry has a deployable version before the demo. Also starts the recurring training scheduler.


### Step 23 — Build trainer image and run baseline + fine-tune (stage1)

The baseline establishes the reference CER that every later candidate's quality gate compares against. Then one fine-tune candidate runs to populate `htr/v1` in the MLflow model registry.

We use `microsoft/trocr-base-stage1` (pretrained on synthetic printed text, never fine-tuned on handwriting) as the starting point. This is the same base Microsoft used internally before fine-tuning on IAM to produce `trocr-base-handwritten` — we replicate that step on a smaller scale to demonstrate an end-to-end training pipeline that actually teaches the model something new.

On CHI `compute_cascadelake` (48 cores, CPU-only): baseline ~3 min, finetune_iam_stage1 ~30-45 min.


In [ ]:
print("Pulling latest code from git (in case of in-flight changes)...")
s.execute("cd ~/paperless_data_integration && git pull")
s.execute("cd ~/paperless_data && git pull")

print("\nBuilding trainer image (~10 min)...")
s.execute(
    "cd ~/paperless_data_integration && "
    "sg docker -c 'docker compose -p training -f training/compose.yml build baseline_stage1'"
)

print("\nRunning baseline_stage1 (~3 min) — records reference CER into MLflow experiment tag...")
s.execute(
    "cd ~/paperless_data_integration && "
    "sg docker -c 'docker compose -p training -f training/compose.yml run --rm baseline_stage1'"
)

print("\nRunning finetune_iam_stage1 (~30-45 min on CHI cascadelake) — should PASS the gate and register as htr/v1...")
s.execute(
    "cd ~/paperless_data_integration && "
    "sg docker -c 'docker compose -p training -f training/compose.yml run --rm finetune_iam_stage1'"
)

print("\nVerifying registry...")
s.execute(
    "curl -s 'http://localhost:5000/api/2.0/mlflow/registered-models/get?name=htr' "
    "| python3 -m json.tool"
)


### Step 24 — Deploy the registered model to ml_gateway

Flips the `current_htr.txt` file in the shared volume to `models:/htr/1` and SIGHUPs ml_gateway. Verifies the reload via the /health endpoint.


In [ ]:
s.execute(
    "cd ~/paperless_data_integration && "
    "chmod +x scripts/deploy_model.sh && "
    "sg docker -c 'bash scripts/deploy_model.sh latest'"
)

print("\nml_gateway health after SIGHUP reload:")
s.execute("curl -s http://localhost:8100/health | python3 -m json.tool")


### Step 25 — Start recurring training scheduler

Long-running daemon that fires `finetune_combined_stage1` every `INTERVAL_MIN` minutes (default 60 for demo visibility; production would use 1440 = daily). The combined-stage1 config pulls from IAM + any user corrections saved via `/ml/htr-review`, re-fine-tunes, and registers a new `htr` version if the quality gate passes.

The scheduler skips firing if a training container is already running — safe to leave on indefinitely.


### Step 26 — Bring up Airflow for orchestration

Orchestrates three feedback loops across the MLOps platform. The `up_airflow.sh` script handles prerequisites — it builds the custom `paperless-airflow:2.10.4` image (includes `apache-airflow-providers-docker` + `psycopg2-binary` + `minio` SDK preinstalled), and also builds the two data images spawned by the HTR training DAG:

- `htr_trainer:latest` — fine-tunes TrOCR on IAM + corrections
- `htr_batch:latest` — builds training snapshots from archived corrections

**DAGs registered:**

| DAG | Schedule | Purpose |
|---|---|---|
| `archive_corrections` | every 15 min | Upload unarchived corrections to MinIO as immutable JSON for audit + reproducibility |
| `search_feedback_rerank` | hourly | Aggregate thumbs-up/down into `document_feedback_stats` for query-time reranking |
| `htr_retraining` | daily 02:00 UTC | `build_snapshot` (reads archive) → `finetune_combined_stage1` → `notify_result` |

The Airflow stack (metadata postgres + init + webserver + scheduler) uses ~2 GB of disk. First-time boot takes ~60s to build the custom Airflow image + init the metadata DB; subsequent boots ~10s.

UI: `http://<vm>:8080` — login `admin` / `admin`


In [ ]:
print("Pulling latest code (airflow/ dir + up_airflow.sh)...")
s.execute("cd ~/paperless_data_integration && git pull && cd ~/paperless_data && git pull")

print("\nBringing up Airflow (metadata DB, init, webserver, scheduler)...")
# First boot builds paperless-airflow:2.10.4 + htr_trainer:latest + htr_batch:latest
s.execute("cd ~/paperless_data_integration && chmod +x scripts/up_airflow.sh && sg docker -c 'bash scripts/up_airflow.sh'")

print("\nAirflow containers:")
s.execute("sg docker -c 'docker compose -p airflow -f ~/paperless_data_integration/airflow/compose.yml ps'")

print("\nRegistered DAGs (after scheduler has parsed the dags/ dir):")
s.execute(
    "for i in $(seq 1 20); do "
    "  result=$(curl -sf -m 3 -u admin:admin http://localhost:8080/api/v1/dags 2>/dev/null); "
    "  if [ -n \"$result\" ]; then "
    "    echo \"$result\" | python3 -c \"import json,sys; d=json.load(sys.stdin); [print(f\"  {x['dag_id']:35s} schedule={x.get('schedule_interval',{}).get('value','-') if isinstance(x.get('schedule_interval'),dict) else x.get('schedule_interval','-')}\") for x in d.get('dags',[])]\" && break; "
    "  fi; sleep 3; "
    "done"
)

# Pull the PUBLIC floating IP, not the fixed internal 10.52.x.x address.
# OpenStack returns addresses as {network_name: [ {addr, OS-EXT-IPS:type}, ... ]}
# where OS-EXT-IPS:type is either "fixed" (internal) or "floating" (public).
_public_ips = [
    a["addr"]
    for _net, addrs in s.addresses.items()
    for a in addrs
    if a.get("OS-EXT-IPS:type") == "floating"
]
fip = _public_ips[0] if _public_ips else "<check s.addresses>"
print(f"\nAirflow UI:       http://{fip}:8080  (admin / admin)")
print(f"DAGs available:   htr_retraining  (daily 02:00 UTC, manual trigger via Play icon)")
print(f"                  search_feedback_rerank  (hourly)")


In [ ]:
s.execute(
    "cd ~/paperless_data_integration && "
    "sg docker -c 'docker compose -p training -f training/compose.yml build training_scheduler'"
)

s.execute(
    "cd ~/paperless_data_integration && "
    "sg docker -c 'INTERVAL_MIN=60 docker compose -p training -f training/compose.yml up -d training_scheduler'"
)

print("\nScheduler status:")
s.execute("sg docker -c 'docker ps --filter name=training_scheduler --format \"table {{.Names}}\\t{{.Status}}\"'")

print("\nFirst scheduler log lines:")
s.execute("sg docker -c 'docker logs training_scheduler 2>&1 | head -20'")


---
## Teardown

Uncomment and run when you're done to release resources.

In [ ]:
# s = server.get_server(f"node-paperless-integration-{username}")
# server.delete_server(s.id)
# l = lease.get_lease(f"lease-paperless-integration-{username}")
# lease.delete_lease(l.id)
# print("Resources released.")